In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rmisra/news-category-dataset")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/news-category-dataset


In [2]:
import pandas as pd 
import numpy as np
import json 
import uuid
import random

In [44]:
# Path to your JSON file
file_path = '/kaggle/input/news-category-dataset/News_Category_Dataset_v3.json'

# Load the JSON file line by line
with open(file_path, 'r') as file:
    data1 = [json.loads(line) for line in file]

# Print the loaded data (optional)
print(data1[:5])  # Print the first 5 entries for review


[{'link': 'https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9', 'headline': 'Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters', 'category': 'U.S. NEWS', 'short_description': 'Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.', 'authors': 'Carla K. Johnson, AP', 'date': '2022-09-23'}, {'link': 'https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe', 'headline': 'American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video', 'category': 'U.S. NEWS', 'short_description': "He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to the U.S. attorney's office in Los Angeles.", 'authors': 'Mary Papenfuss', 'date': '2022-09-23'}, {'link': 'https://www.huffpost.com/en

In [45]:
data = data1[:50000] 

In [48]:
# Create DataFrame 1 (df1) with link, headline, category, short_desc, and date
df1 = pd.DataFrame([{
    'link': item.get('link'),
    'headline': item.get('headline'),
    'category': item.get('category'),
    'short_desc': item.get('short_description'),
    'date': item.get('date'),
    'authors': item.get('authors')
} for item in data])

# Create DataFrame 2 (df2) with author
df2 = pd.DataFrame([{
    'authors': item.get('authors')
} for item in data])

In [49]:
def separate_values_to_new_lines(df, column_name, delimiter=','):    
    # Create a new DataFrame by expanding the values in the specified column
    df_separated = df.explode(column_name).reset_index(drop=True)
    
    # Strip any leading/trailing spaces after splitting
    df_separated[column_name] = df_separated[column_name].str.strip()
    
    return df_separated

In [ ]:
print(df1.info())
print(df2.info())

In [50]:
df2 = df2.apply(lambda col: col.apply(lambda x: str(x) if x != '' else np.nan))
df2 = separate_values_to_new_lines(df2, 'authors')
df2 = df2.dropna()
df2 = df2.drop_duplicates()
df1 = df1.dropna(subset=['authors'])
df1 = df1.drop_duplicates()

In [51]:
def generate_unique_id(uniq):
    return uniq + str(random.randint(100000, 999999))
# artikel awali dengan 1010
# user awali dengan 2020
# Menambahkan kolom dengan ID unik
df1['id_artikel'] = df1.apply(lambda x: generate_unique_id("1010"), axis=1)
df2['id_user'] = df2.apply(lambda x: generate_unique_id("2020"), axis=1)


In [52]:
# Daftar region di USA
usa_regions = ['Northeast', 'Midwest', 'South', 'West']

# Fungsi untuk menghasilkan region USA acak
def generate_random_region():
    return random.choice(usa_regions)

# Fungsi untuk menghasilkan umur acak
def generate_random_age(min_age=20, max_age=55):
    return random.randint(min_age, max_age)

# Menambahkan kolom region dan age dengan nilai acak
df2['region'] = [generate_random_region() for _ in range(len(df2))]
df2['age'] = [generate_random_age() for _ in range(len(df2))]

In [53]:
print(df1.info())
print(df2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   link        50000 non-null  object
 1   headline    50000 non-null  object
 2   category    50000 non-null  object
 3   short_desc  50000 non-null  object
 4   date        50000 non-null  object
 5   authors     50000 non-null  object
 6   id_artikel  50000 non-null  object
dtypes: object(7)
memory usage: 2.7+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 7555 entries, 0 to 49986
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   authors  7555 non-null   object
 1   id_user  7555 non-null   object
 2   region   7555 non-null   object
 3   age      7555 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 295.1+ KB
None


In [54]:
df1.to_csv("vector_dataFinal.csv", index=False)

In [35]:
def generate_user_arc(df1, df2):
    # Prepare users DataFrame (df1) with the required columns
    users_data = df1[['authors', 'id_user', 'region', 'age']].rename(columns={
        'authors': 'name',
        'id_user': 'id_user',
        'region': 'region',
        'age': 'age'
    }).to_dict(orient='records')
    
    # Prepare articles DataFrame (df2) with the required columns
    articles_data = df2[['link', 'headline', 'category', 'date', 'id_artikel']].rename(columns={
        'id_artikel': 'id_arc',
        'link': 'link',
        'headline': 'headline',
        'category': 'category',
        'date': 'date'
    }).to_dict(orient='records')
    return users_data, articles_data
u, a = generate_user_arc(df2, df1)

In [55]:
graphuser = df2.copy()

In [56]:
graphuser.to_csv("graphuserFinal.csv", index=False)

In [57]:
grapharc = df1.drop(columns=["short_desc", "authors"]).copy()

In [58]:
grapharc.to_csv("grapharcFinal.csv", index=False)

In [62]:
df1['authors'] = df1['authors'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
df2['authors'] = df2['authors'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

In [63]:
merged_df = pd.merge(df1, df2, on='authors', how='inner')

In [65]:
merged_df[["id_artikel", "id_user"]].to_csv("graphrelaFinal.csv", index=False)